# Test various HIPS maps

In [ ]:
from astropy.wcs import WCS
from astroquery.hips2fits import hips2fits
from astropy.io import fits
import astropy.units as u
from astropy.visualization import ImageNormalize, AsinhStretch, PercentileInterval
import matplotlib.pyplot as plt

In [ ]:
VALID_HIPS = [
    "CDS/P/DSS2/color",
    "CDS/P/DSS2/red",
    "CDS/P/DSS2/blue",
    "CDS/P/PanSTARRS/DR1/r",
    "CDS/P/PanSTARRS/DR1/g",
    "CDS/P/PanSTARRS/DR1/i",
    "CDS/P/PanSTARRS/DR1/z",
    "CDS/P/PanSTARRS/DR1/y",
    "CDS/P/DES-DR2/ColorIRG",
    "CDS/P/DES-DR2/r",
    "CDS/P/DES-DR2/i",
    "CDS/P/DES-DR2/g",
]

CANDIDATE_HIPS = ["CDS/P/PanSTARRS/DR1/r", "CDS/P/DES-DR2/r", "CDS/P/DSS2/color"]

In [ ]:
index_selected = 3
hips_selected = VALID_HIPS[index_selected]

In [ ]:
wcs = WCS(naxis=2)

wcs.wcs.crpix = [128, 128]
wcs.wcs.crval = [150.1191, 2.2058]
wcs.wcs.cdelt = [-0.00011, 0.00011]
wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"]

# 🔴 IMPORTANT : taille de l’image
wcs.array_shape = (256, 256)

hdul = hips2fits.query_with_wcs(
    # hips="CDS/P/PanSTARRS/DR1/r",
    hips=hips_selected,
    wcs=wcs,
    # format="fits",
    # get_query_payload=True
)

In [ ]:
hdul = hips2fits.query(
    # hips ="CDS/P/PanSTARRS/DR1/r",
    hips=hips_selected,
    ra=150.1191 * u.deg,
    dec=2.2058 * u.deg,
    fov=0.05 * u.deg,
    width=256,
    height=256,
    projection="TAN",
)

In [ ]:
print(type(hdul))
print(hdul.info())

In [ ]:
data = hdul[0].data
hdr = hdul[0].header

In [ ]:
print(data.shape, data.dtype)

In [ ]:
wcs = WCS(hdr).celestial

In [ ]:
fig = plt.figure(figsize=(10, 10), layout="constrained")
ax = fig.add_subplot(111, projection=wcs)

ax.imshow(data, origin="lower", cmap="gray")
ax.set_xlabel("RA")
ax.set_ylabel("Dec")

ax.grid()


overlay = ax.get_coords_overlay("fk5")
overlay[0].set_axislabel("RA (deg)")
overlay[1].set_axislabel("Dec (deg)")
overlay[0].set_format_unit(u.deg)
overlay[1].set_format_unit(u.deg)

overlay[0].set_major_formatter("d.dd")  # degrés avec décimales
overlay[1].set_major_formatter("d.dd")


overlay.grid(color="red", ls="dotted")
plt.suptitle(f" hips : {hips_selected}", y=1.005)
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection=wcs)

ax.imshow(data, origin="lower", cmap="gray")

# Accès aux coordonnées WCS
ra = ax.coords[0]
dec = ax.coords[1]

ra.set_format_unit(u.deg)
dec.set_format_unit(u.deg)

ra.set_major_formatter("d.dd")  # degrés avec décimales
dec.set_major_formatter("d.dd")


ra.set_axislabel("RA [deg]")
dec.set_axislabel("Dec [deg]")

ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
def ShowHipsImage(wcs, data, fig=None):
    """ """

    if fig is None:
        fig = plt.figure(figsize=(10, 10), layout="constrained")

    norm = ImageNormalize(
        data,
        interval=PercentileInterval(99.99),  # coupe les extrêmes
        stretch=AsinhStretch(),  # non-linéaire doux
    )

    ax = fig.add_subplot(111, projection=wcs)
    ax.imshow(data, origin="lower", cmap="gray", norm=norm)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid()

    overlay = ax.get_coords_overlay("fk5")
    overlay[0].set_axislabel("RA (deg)")
    overlay[1].set_axislabel("Dec (deg)")
    overlay[0].set_format_unit(u.deg)
    overlay[1].set_format_unit(u.deg)
    overlay[0].set_major_formatter("d.dd")  # degrés avec décimales
    overlay[1].set_major_formatter("d.dd")
    overlay.grid(color="red", ls="dotted")
    return ax

## Loop

In [ ]:
ax = ShowHipsImage(wcs, data)
plt.suptitle(f" hips : {hips_selected}", y=1.005)
plt.show()

In [ ]:
all_hdul = []

for hips_tested in VALID_HIPS:
    try:
        hdul = hips2fits.query_with_wcs(
            hips=hips_tested,
            wcs=wcs,
            # format="fits",
            # get_query_payload=True
        )

        hdul = hips2fits.query(
            hips=hips_tested,
            ra=150.1191 * u.deg,
            dec=2.2058 * u.deg,
            fov=0.05 * u.deg,
            width=256,
            height=256,
            projection="TAN",
        )

        all_hdul.append(hdul)
        print(f">>>>> SUCCESS {hips_tested}")
        print(hdul.info())

    except:
        print(f"+++++ FAILED {hips_tested}")

In [ ]:
for idx, hips_tested in enumerate(VALID_HIPS):
    hdul = all_hdul[idx]

    print(hdul.info())

    if hdul is None:
        print(f"+++++ Skip {hips_tested }")
        continue

    try:
        data = hdul[0].data
        hdr = hdul[0].header
        wcs = WCS(hdr)

        if wcs.pixel_n_dim > 2:
            wcs_plot = wcs.celestial
        else:
            wcs_plot = wcs

        fig = plt.figure(figsize=(8, 8), layout="constrained")
        ax = ShowHipsImage(wcs_plot, data, fig)
        plt.suptitle(f" hips : {hips_tested}", y=1.005)
        plt.show()
    except:
        print(f"+++++ FAILED plotting {hips_tested}")